# Activity 2: Implementing Iterative Proportional Fitting

In this notebook, you will implement a classical version of the IPF algorithm, as well as some aspects of our variation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

For reading, note that you might know IPF as:

- Iterative proportional fitting
- Raking
- IPFP (the p is for procedure...)
- Biproportional sacaling
- RAS algorithm

## 1. Implementing the Classical IPF

Classical IPF proposes that we repeatedly rescale rows and columns of a matrix so that they match known row and column sum constraints. That is, we would like to estimate an $n\times m$-dimensional matrix $A$ and we are given:

- A noisy $n\times m$-dimensional matrix $B$ (for example, a perturbation of $A$, or a sample of $A$).
- An $n$-dimensional vector $X$ which represents the sums of the rows of $A$. That is, the sum of the first row of $A$ is $X_1$, the sum of the second row is $X_2$, etc.
- An $m$-dimensional vector $Y$ which represents the sums of the columns of $A$.  That is, the sum of the first column of $A$ is $Y_1$, etc.

For example, if we knew $A$ was:

In [ ]:
A = np.array([[40, 50, 12, 30],
              [20, 30, 50, 10],
              [30, 20, 10, 40],
              [10, 40, 30, 20],
              [5, 40, 30, 20]])

Question: What are $n$ and $m$? What are $X$ and $Y$? Define them below:

In [ ]:
n = None
m = None
X = None #np.array([...])
Y = None #np.array([...])

#Can you compute the row and column sums of A to produce X and Y?

Now, forget you ever saw the matrix $A$! You only saw a matrix $B$, which is:

In [ ]:
B = np.array([[8, 11, 2, 1],
              [5, 20, 3, 2],
              [6, 1, 2, 6],
              [2, 8, 6, 4],
              [4, 7, 5, 8]], dtype=float) #Note that we need to use floats here, otherwise the multipliers below will be rounded to integers!
B0 = B.copy() #Keep a copy of the initial matrix for comparison later

Complete the function above to apply the iterative proportional fitting algorithm to $B$ and estimate $A$:

In [ ]:
n_iterations = 10

#This version uses for loops over the rows and columns---feel free to vectorize!
# Be careful with the dimensions of the arrays! Use print statements to check them if you are unsure.
for iteration in range(n_iterations):
    print(f'\nIteration {iteration+1}')

    #Rescale the rows of B to match the row sums X:
    current_row_sums_in_B = None #a 1-dimensional numpy array of the current row sums of B
    for i in range(n):
        multiplier = None #the multiplier to rescale row i of B to match X[i], which will depend on current_row_sums_in_B[i] and X[i]
        B[i, :] = B[i, :] * multiplier
    
    #Rescale the columns of B to match the column sums Y:
    current_col_sums_in_B = None #a 1-dimensional numpy array of the current column sums of B
    for j in range(m):
        multiplier = None #the multiplier to rescale col j of B to match Y[j], which will depend on current_col_sums_in_B[j] and Y[j]
        B[:, j] = B[:, j] * multiplier


    #Print the difference between the row sum of B and X, and the difference between the column sum of B and Y
    print(abs(row_sums_of_B_after_iteration - X)) #Should be close to X
    print(abs(col_sums_of_B_after_iteration - Y)) #Should be close to Y
    #Can you visualize the average of these errors over the iterations with plt.plot()? You may need to store them in lists.

    break #remove this when you have checked the first iteration

Compare $A$ and (the inferred) $B$

In [ ]:
A

In [ ]:
B

### IPF in survey weighting

One very useful property of IPF is that when it converges the solution is unique. And it can be used in survey weighting---something you may find very useful!

- Let's say you surveyed a population of 537 people.
- Each person was classified in one of 4 income groups (1st quartile, 2nd quartile, 3rd quartile, 4th quartile)
- Each person was classified in one of 5 age groups (Under 18, 18-30, 30-50, 50-65, Above 65)

You know the total number of people in each age group per each income group via the Census---assume that's our matrix A

In [ ]:
A_as_dataframe = pd.DataFrame(A, columns=['Income 1st quartile', 'Income 2nd quartile', 'Income 3rd quartile', 'Income 4th quartile'],
                              index=['Age < 18', 'Age 18-30', 'Age 30-50', 'Age 50-65', 'Age > 65'])
display(A_as_dataframe)

Then you do IPF in the matrix from your sample---for example, if $B$ was the original sample (which we saved as $B0$), we have sampled:

In [ ]:
B_as_dataframe = pd.DataFrame(B0, columns=['Income 1st quartile', 'Income 2nd quartile', 'Income 3rd quartile', 'Income 4th quartile'],
                 index=['Age < 18', 'Age 18-30', 'Age 30-50', 'Age 50-65', 'Age > 65'])
display(B_as_dataframe)

The weight of an individual in a survey is the relative value of that individual's response. If your sample is truly random, all weights should be equal. But often you may find that you oversampled some population, or undersampled some population. IPF ``corrected`` your sample, producing weights for each individaul---the ratio of the inferred to the original values:

In [ ]:
w_as_dataframe = pd.DataFrame(B/B0, columns=['Income 1st quartile', 'Income 2nd quartile', 'Income 3rd quartile', 'Income 4th quartile'],
                 index=['Age < 18', 'Age 18-30', 'Age 30-50', 'Age 50-65', 'Age > 65'])
display(w_as_dataframe.round(2))

So for example the weights reveal that you would need to value the responses of a 3rd-income quartile 17 year old ($w=10.77$) over twice as much as those of a 1st income quartile 17 year old ($w=5.25$). Note that weights are all greater than 1 because $B$ at first summed to a value much below $A$---to actually understand representation, you could have rescaled $B$ by multiplying every entry by $A.sum()/B.sum()$.

For further reading, and especially if you are truly interested in using IPF in sample weighting, there exist more advanced methods to do IPF which also try to minimize other survey problems. For example, weights that are close to uniform tend to be better---to avoid disproportionately valuing a few responses. Generalized raking solves this issue, here is a [nice read](https://dev.to/potloc/generalized-raking-for-survey-weighting-2d1d) with code snippets.

## 2. Our variation of IPF

Now that we know how to do the classical IPF variation, let's think about our version of the procedure. Here I will give you a CBG-CBG noisy matrix $M$ and many constraints (which are not exactly the ones we have on the paper, feel free to ask me what are the differences!), as well as <b> membership matrices </b> $C$. These matrices $C$ are binary and tell you to what county and state each CBG belongs to.

In [ ]:
#Assume that M is a noisy estimate of the true CBG-CBG migration matrix between 2015 and 2016.
M = np.array([[30,  2, 1,  0,   4,  5,  8],
              [10, 50, 5,  2,   1,  0,  3],
              [15,  3, 40, 8,   0,  2,  1],
              [0,   1, 4,  30,  5,  3,  2],
              [2,   0, 1,  5,  20,  4,  3],
              [1,   2, 0,  3,   4, 25,  5],
              [0,   1, 2,  1,   3,  5, 15]], dtype=float)
M0 = M.copy() #Keep a copy of the initial matrix for comparison later

#Constraints:
CBG_population_2015 = np.array([100, 30, 60, 50, 80, 19, 25]) #The population of each CBG in 2015
CBG_population_2016 = np.array([120, 10, 60, 70, 90, 10, 4])  #The population of each CBG in 2016

county_population_2015 = np.array([130, 190, 44]) #The population of each county in 2015
county_population_2016 = np.array([130, 220, 14]) #The population of each county in 2016

state_nonmovers = np.array([100, 150])
state_to_state_movers = np.array([[121, 41],
                                  [21, 181]])

#Membership matrices:
C_county = np.array([[1, 0, 0],  #CBG 0 is in county 0
                     [1, 0, 0],  #CBG 1 is in county 0
                     [0, 1, 0],  #CBG 2 is in county 1
                     [0, 1, 0],  #CBG 3 is in county 1
                     [0, 1, 0],  #CBG 4 is in county 1
                     [0, 0, 1],  #CBG 5 is in county 2
                     [0, 0, 1]]) #CBG 6 is in county 2
C_state = np.array([[1, 0],  #CBG 0 is in state 0
                    [1, 0],  #CBG 1 is in state 0
                    [0, 1],  #CBG 2 is in state 0
                    [0, 1],  #CBG 3 is in state 0
                    [0, 1],  #CBG 4 is in state 1
                    [0, 1],  #CBG 5 is in state 1
                    [0, 1]]) #CBG 6 is in state 1

Here are some of our sacalings, which we do <b> only once </b> i.e. we do not iterate over them --- we do that to prevent overfitting!

1. We scale rows of the CBG-CBG flow matrix to match CBG populations at the <b>initial or final?</b> time i.e. everyone who left a particular CBG (off-diagonal entries) or stayed in that CBG (diagonal entry) should add up to the CBG population at the correct time.

In [ ]:
#Decide on initial or final, and do that scaling. I recommend calling the scaled matrix M_scaled!
current_CBG_row_sums = M.sum(axis=1)
M = None #very similar to IPF!

This will be considerably harder! You may want to use pencil or paper, or ask for help

2. We scale diagonal entries of the CBG-CBG matrix to match data on the population that <b> did not move </b> per state.

We have data of people who did not move at the state level (`state_nonmovers`). So we need to:
- Group diagonal entries of $M$ according to the state they belong to.
- Scale these groups to match the data on `state_nonmovers`

Note: this scaling hides an assumption---the number of people moving within their own CBG is negligible (do you see that? and do you agree with that?).

Note2: we also scale folks who move (columns) to match the difference between state non-movers and state populations. Feel free to try that!

In [ ]:
current_CBG_nonmovers = None #Get the current number of people who do not move CBG
current_state_nonmovers = None #Use the matrix C_state to sum current_CBG_nonmovers per state. Hint: matrix multiplication!
scalers_per_state = None #We need to find scalers such that current_state_nonmovers * scalers_per_state = state_nonmovers
#Use C_state again to assign to each CBG the scaler of the state it belongs to
#Finally, scale the diagonal of M_scaled with these CBG scalers

We also iteratively match the population at the county level. That is, we do IPF but instead of matching every row and every column, we match every GROUP of rows and every GROUP of columns. Would you like to take a shot?